# Phase 2: Random Forest Pipeline (`rf_pipeline.joblib`)

The Phase 1 Random Forest expects the 102 **preprocessed** columns, but the dashboard receives **raw** student values (for example a grade of 13.5, not a scaled value). This notebook joins the Phase 1 preprocessing and the tuned Random Forest into one scikit-learn `Pipeline`, checks that it gives exactly the same results as Phase 1, and saves it.

- **Input:** a raw student record with the same columns as `Dataset/data.csv` (2nd-semester columns and `Target` are ignored if present).
- **Output:** `Supervised_Learning/models/rf_pipeline.joblib`
- **Used by:** `pages/1_Student_Simulator.py`, the GenAI briefings, and the cohort pages.

```python
from Supervised_Learning.rf_pipeline_utils import load_rf_pipeline
rf_pipeline = load_rf_pipeline()
proba = rf_pipeline.predict_proba(raw_student_df)   # columns: Dropout, Enrolled, Graduate
```

In [ ]:
# Google Colab only: upload the files this notebook needs (skipped when they already exist)
from pathlib import Path

needed = {
    "data.csv": Path("Dataset/data.csv"),
    "preprocessed_data.csv": Path("Dataset/preprocessed_data.csv"),
    "rf_pipeline_utils.py": Path("Supervised_Learning/rf_pipeline_utils.py"),
}
missing = {name: path for name, path in needed.items() if not path.exists()}

if missing:
    try:
        from google.colab import files
        print("Upload:", ", ".join(missing))
        uploaded = files.upload()
        for name, path in missing.items():
            match = next((k for k in uploaded if k.startswith(Path(name).stem)), None)
            if match is None:
                raise FileNotFoundError(f"Please upload {name}.")
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_bytes(uploaded[match])
    except ImportError:
        raise FileNotFoundError(f"Missing files: {list(missing.values())}")

print("All files ready.")


## 1. Setup
Imports the preprocessing class from `Supervised_Learning/rf_pipeline_utils.py`. The pipeline file stores a reference to this module, so every page that loads `rf_pipeline.joblib` must run from the repository root, where `Supervised_Learning.rf_pipeline_utils` can be imported.

In [ ]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

from Supervised_Learning.rf_pipeline_utils import (
    CLASS_NAMES, TARGET_CODES, StudentPreprocessor, load_raw_data, add_engineered_features,
)

RANDOM_STATE = 42
import sklearn
print("scikit-learn", sklearn.__version__, "(Phase 1 results were produced with 1.6.1)")
models_dir = Path("Supervised_Learning/models")
models_dir.mkdir(parents=True, exist_ok=True)


## 2. Raw data and the Phase 1 split
The pipeline is trained on the **raw** dataset (`Dataset/data.csv`), so it can accept raw student inputs in the dashboard. The split repeats Phase 1 exactly: 80/20, stratified, `random_state=42`.

In [ ]:
raw = load_raw_data("Dataset/data.csv")
y = raw["Target"].map(TARGET_CODES)
X_raw = raw.drop(columns="Target")

# Same split as Phase 1: split rows AFTER the same column steps, 80/20, stratified, random_state=42
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train:", X_train_raw.shape, "| Test:", X_test_raw.shape)


## 3. Tuned Random Forest settings
No new tuning is done here. The pipeline reuses the Phase 1 tuned settings (200 trees, `max_depth=None`, `min_samples_split=10`, `class_weight="balanced"`, `random_state=42`).

In [ ]:
# Tuned settings from Phase 1 (GridSearchCV, 5-fold, f1_macro). Read them from the saved
# Phase 1 model when it exists, so the two can never drift apart.
phase1_model_path = models_dir / "random_forest_tuned.joblib"
if phase1_model_path.exists():
    rf_params = joblib.load(phase1_model_path).get_params()
    print("Using parameters from", phase1_model_path)
else:
    rf_params = dict(n_estimators=200, max_depth=None, min_samples_split=10,
                     class_weight="balanced", random_state=RANDOM_STATE)
    print("Phase 1 model not found, using the documented tuned parameters")


## 4. Build the pipeline
`StudentPreprocessor` repeats the Phase 1 preprocessing: drop the 2nd-semester columns, add `academic_progress` and `no_units_enrolled`, group categories used by less than 1% of training students into `-1`, one-hot encode, and standard-scale the 13 numeric columns. All of these are learned from the training students only.

In [ ]:
# Phase 1 fitted the preprocessing on the training rows in train_test_split order,
# but trained the Random Forest on the rows in their original order (sorted index,
# as read back from preprocessed_data.csv). Row order changes the bootstrap samples,
# so the two steps are fitted the same way here and then joined into one Pipeline.
preprocessor = StudentPreprocessor(rare_threshold=0.01).fit(X_train_raw)

X_train_sorted = X_train_raw.sort_index()
y_train_sorted = y_train.sort_index()
rf_model = RandomForestClassifier(**rf_params).fit(
    preprocessor.transform(X_train_sorted), y_train_sorted
)

rf_pipeline = Pipeline([("preprocess", preprocessor), ("model", rf_model)])


## 5. Check 1: same features as Phase 1
The pipeline's preprocessing output is compared with `Dataset/preprocessed_data.csv` for every student. Differences around `1e-15` are normal floating-point rounding.

In [ ]:
pre = pd.read_csv("Dataset/preprocessed_data.csv")
phase1_X = pre.drop(columns=["Target", "split"])
pipe_X = pd.concat([
    rf_pipeline.named_steps["preprocess"].transform(X_train_raw),
    rf_pipeline.named_steps["preprocess"].transform(X_test_raw),
]).sort_index()

same_split = (pre.loc[X_test_raw.index, "split"] == "test").all() and \
             (pre.loc[X_train_raw.index, "split"] == "train").all()
same_columns = list(pipe_X.columns) == list(phase1_X.columns)
max_diff = np.abs(pipe_X.values.astype(float) - phase1_X.values.astype(float)).max()
print("Same train/test split as Phase 1 :", same_split)
print("Same 102 columns in same order   :", same_columns)
print("Largest feature difference       :", max_diff)


## 6. Check 2: same predictions as Phase 1
The pipeline's test-set predictions and probabilities are compared with the Phase 1 Random Forest. They must be identical.

*Note:* Random Forest results can change between scikit-learn versions. The numbers must be compared within the same environment; the Phase 1 figures (accuracy 74.1%, macro F1 0.673) were produced with scikit-learn 1.6.1 (`requirements.txt`).

In [ ]:
# Reference = the Phase 1 Random Forest. Use the saved Phase 1 model when it exists;
# otherwise retrain it exactly as Phase 1 did (preprocessed_data.csv, same parameters).
train_p1, test_p1 = pre[pre["split"] == "train"], pre[pre["split"] == "test"]
X_test_p1 = test_p1.drop(columns=["Target", "split"])
if phase1_model_path.exists():
    phase1_rf = joblib.load(phase1_model_path)
else:
    phase1_rf = RandomForestClassifier(**rf_params).fit(
        train_p1.drop(columns=["Target", "split"]), train_p1["Target"]
    )
p1_pred = phase1_rf.predict(X_test_p1)
p1_proba = phase1_rf.predict_proba(X_test_p1)

pipe_pred = rf_pipeline.predict(X_test_raw.sort_index())
pipe_proba = rf_pipeline.predict_proba(X_test_raw.sort_index())
y_test_sorted = y_test.sort_index()

print("Identical predictions            :", bool((pipe_pred == p1_pred).all()))
print("Largest probability difference   :", np.abs(pipe_proba - p1_proba).max())
print(f"Phase 1 model  - accuracy {accuracy_score(test_p1['Target'], p1_pred):.4f} | "
      f"macro F1 {f1_score(test_p1['Target'], p1_pred, average='macro'):.4f}")
print(f"rf_pipeline    - accuracy {accuracy_score(y_test_sorted, pipe_pred):.4f} | "
      f"macro F1 {f1_score(y_test_sorted, pipe_pred, average='macro'):.4f}")
print(classification_report(y_test_sorted, pipe_pred, target_names=CLASS_NAMES, digits=4))


## 7. Save `rf_pipeline.joblib`
The file is saved only if both checks pass. A reload test on one raw student confirms the saved file works on its own.

In [ ]:
assert same_split and same_columns and max_diff < 1e-9 and (pipe_pred == p1_pred).all(), \
    "Pipeline does not match Phase 1 - do not save"
joblib.dump(rf_pipeline, models_dir / "rf_pipeline.joblib", compress=3)

# Reload and test on one raw student to confirm the saved file works
reloaded = joblib.load(models_dir / "rf_pipeline.joblib")
one_student = X_test_raw.iloc[[0]]
print("Reloaded pipeline, one raw student ->",
      CLASS_NAMES[reloaded.predict(one_student)[0]],
      reloaded.predict_proba(one_student).round(3))
print("Saved:", models_dir / "rf_pipeline.joblib",
      f"({(models_dir / 'rf_pipeline.joblib').stat().st_size / 1e6:.2f} MB)")


## 8. Download the pipeline (Google Colab only)

In [ ]:
try:
    from google.colab import files
    files.download("Supervised_Learning/models/rf_pipeline.joblib")
except ImportError:
    print("Not running in Colab: the file is already saved in Supervised_Learning/models/.")
